# SkyFinder Temperature Prediction — Training Notebook

Run on **Google Colab with T4 GPU**.

Steps:
1. Mount Drive, clone/pull repo
2. Restore images from Drive tarball
3. Run `prepare_data.py`
4. Smoke test (subset, 1 epoch)
5. Vanilla baseline (~15 epochs)
6. Metadata-only baseline for comparison

In [ ]:
# === Settings ===
STORE_ROOT = "/content/drive/MyDrive/skyfinder_runs"
GITHUB_USER = "alexanderlio"
REPO_NAME = "imbalanced-regression-on-skyfinder"

In [ ]:
# === Mount Drive + Clone/Pull Repo ===
from google.colab import drive
drive.mount("/content/drive")

import os
repo_dir = f"/content/{REPO_NAME}"
if os.path.isdir(repo_dir):
    !cd {repo_dir} && git pull
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {repo_dir}

os.chdir(repo_dir)
!pip install -q scipy tqdm tensorboard_logger

In [ ]:
# === Restore images tarball from Drive ===
import tarfile

tarball = "/content/drive/MyDrive/skyfinder_images.tar.gz"
img_dir = "data/images"
if not os.path.isdir(img_dir):
    print(f"Extracting {tarball} ...")
    with tarfile.open(tarball, "r:gz") as tar:
        tar.extractall("data")
    print("Done.")
else:
    print(f"{img_dir} already exists, skipping extraction.")

!ls data/images | head -10

In [ ]:
# === Fetch metadata CSV + run prepare_data.py ===
# Download the master CSV if not present
csv_path = "data/complete_table_with_mcr.csv"
if not os.path.isfile(csv_path):
    !wget -q -O {csv_path} "https://zenodo.org/records/5884485/files/complete_table_with_mcr.csv"

!python scripts/prepare_data.py
!head -3 data/skyfinder.csv && echo "..." && wc -l data/skyfinder.csv

In [ ]:
# === Metadata baseline ===
!python scripts/metadata_baseline.py --csv data/skyfinder.csv

In [ ]:
# === Smoke test (1 epoch, small subset) ===
!cd dir_skyfinder && python train.py \
    --data_dir ../data/images \
    --csv ../data/skyfinder.csv \
    --epoch 1 \
    --batch_size 64 \
    --lr 1e-4 \
    --workers 2 \
    --subset_frac 0.05 \
    --img_size 224 \
    --store_root {STORE_ROOT} \
    --store_name smoke \
    --seed 0

In [ ]:
# === Vanilla baseline (15 epochs) ===
!cd dir_skyfinder && python train.py \
    --data_dir ../data/images \
    --csv ../data/skyfinder.csv \
    --epoch 15 \
    --lr 1e-4 \
    --schedule 10 13 \
    --batch_size 64 \
    --workers 2 \
    --img_size 224 \
    --store_root {STORE_ROOT} \
    --seed 0

In [ ]:
# === Resume training (uncomment to use) ===
# !cd dir_skyfinder && python train.py \
#     --data_dir ../data/images \
#     --csv ../data/skyfinder.csv \
#     --epoch 15 \
#     --lr 1e-4 \
#     --schedule 10 13 \
#     --batch_size 64 \
#     --workers 2 \
#     --img_size 224 \
#     --store_root {STORE_ROOT} \
#     --seed 0 \
#     --resume {STORE_ROOT}/skyfinder_resnet50_adam_l1_0.0001_64_seed0/ckpt.best.pth.tar

In [ ]:
# === Print test metrics from each run ===
import json, pathlib

for metrics_path in sorted(pathlib.Path(STORE_ROOT).rglob("test_metrics.json")):
    print(f"\n{'='*60}")
    print(f"Run: {metrics_path.parent.name}")
    print(f"{'='*60}")
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))